# Daily Challenge: Stock Price Prediction with LSTM
## Week 6 — Day 3 | DI GenAI & Machine Learning Bootcamp 2026

**Objective:** Build and train an LSTM model in **PyTorch** to predict stock prices.

**Steps:**
1. Install Required Libraries
2. Load & Preprocess the Dataset
3. Prepare Dataset for Training (custom Dataset + DataLoader)
4. Define the LSTM Model
5. Train the Model
6. Evaluate the Model (R² score + visualizations)

> **Run on Google Colab** — GPU optional but recommended for faster training.

## Step 1 — Install Required Libraries

In [ ]:
!pip install yfinance scikit-learn --quiet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
import pickle

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"PyTorch     {torch.__version__} ✓")
print(f"NumPy       {np.__version__} ✓")
print(f"Pandas      {pd.__version__} ✓")
print(f"Device      : {device}")

torch.manual_seed(42)
np.random.seed(42)

## Step 2 — Load & Preprocess the Dataset

In [ ]:
# Download Apple stock data (2015–2024)
TICKER     = 'AAPL'
START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'

print(f"Downloading {TICKER} stock data from Yahoo Finance...")
raw = yf.download(TICKER, start=START_DATE, end=END_DATE, progress=False)

print(f"\nRaw data shape: {raw.shape}")
print(raw.head())

# Flatten MultiIndex columns if present (yfinance ≥ 0.2.x)
if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(0)

# Drop unnecessary columns — keep OHLCV features
df = raw[['Open', 'High', 'Low', 'Close', 'Volume']].copy()

# Create target: next day's Close price
df['Target'] = df['Close'].shift(-1)

# Drop last row (NaN target) and any other NaNs
df.dropna(inplace=True)

print(f"\nProcessed shape: {df.shape}")
print(f"Date range: {df.index.min().date()}  →  {df.index.max().date()}")
print(f"\nFirst 5 rows:")
display(df.head())

# Visualize raw Close price
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df.index, df['Close'], color='#4e91d4', lw=1)
axes[0].set_title(f'{TICKER} — Closing Price (2015–2024)', fontweight='bold')
axes[0].set_ylabel('Price (USD)'); axes[0].grid(True, alpha=0.3)

axes[1].bar(df.index, df['Volume'], color='#e05c5c', alpha=0.6, width=1)
axes[1].set_title('Trading Volume', fontweight='bold')
axes[1].set_ylabel('Volume'); axes[1].set_xlabel('Date')
axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} Stock Data', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('stock_plot1_raw_data.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 1 saved ✓")

In [ ]:
# Separate features and target, then normalize with MinMaxScaler
feature_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
target_col   = 'Target'

features = df[feature_cols].values
targets  = df[[target_col]].values

scaler_X = MinMaxScaler(feature_range=(0, 1))
scaler_y = MinMaxScaler(feature_range=(0, 1))

features_scaled = scaler_X.fit_transform(features)
targets_scaled  = scaler_y.fit_transform(targets)

print("Normalization ✓")
print(f"  Features range: [{features_scaled.min():.3f}, {features_scaled.max():.3f}]")
print(f"  Targets  range: [{targets_scaled.min():.3f}, {targets_scaled.max():.3f}]")

# Save scalers for future inference
with open('scaler_X.pkl', 'wb') as f: pickle.dump(scaler_X, f)
with open('scaler_y.pkl', 'wb') as f: pickle.dump(scaler_y, f)
print("Scalers saved ✓")

## Step 3 — Prepare Dataset for Training

In [ ]:
LOOK_BACK  = 30   # use past 30 trading days to predict next day
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
# remaining 0.15 → test

n = len(features_scaled)
train_end = int(n * TRAIN_RATIO)
val_end   = int(n * (TRAIN_RATIO + VAL_RATIO))

X_all = features_scaled
y_all = targets_scaled.flatten()

X_train_raw, y_train_raw = X_all[:train_end],  y_all[:train_end]
X_val_raw,   y_val_raw   = X_all[train_end:val_end], y_all[train_end:val_end]
X_test_raw,  y_test_raw  = X_all[val_end:],    y_all[val_end:]

print(f"Split (look_back={LOOK_BACK}):")
print(f"  Train : {len(X_train_raw):,} rows  →  {train_end}")
print(f"  Val   : {len(X_val_raw):,} rows")
print(f"  Test  : {len(X_test_raw):,} rows")

# --- Custom PyTorch Dataset ---
class StockDataset(Dataset):
    def __init__(self, features, targets, look_back):
        self.X = torch.tensor(features, dtype=torch.float32)
        self.y = torch.tensor(targets,  dtype=torch.float32)
        self.look_back = look_back

    def __len__(self):
        return len(self.X) - self.look_back

    def __getitem__(self, idx):
        x_seq = self.X[idx : idx + self.look_back]       # (look_back, n_features)
        y_val = self.y[idx + self.look_back]              # scalar
        return x_seq, y_val

train_dataset = StockDataset(X_train_raw, y_train_raw, LOOK_BACK)
val_dataset   = StockDataset(X_val_raw,   y_val_raw,   LOOK_BACK)
test_dataset  = StockDataset(X_test_raw,  y_test_raw,  LOOK_BACK)

# --- DataLoaders ---
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  drop_last=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Inspect one batch
x_sample, y_sample = next(iter(train_loader))
print(f"\nSample batch:")
print(f"  x shape : {x_sample.shape}  (batch, look_back, n_features)")
print(f"  y shape : {y_sample.shape}  (batch,)")

## Step 4 — Define the LSTM Model

In [ ]:
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super(StockLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers  = num_layers

        self.lstm = nn.LSTM(
            input_size  = input_size,
            hidden_size = hidden_size,
            num_layers  = num_layers,
            batch_first = True,
            dropout     = dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(hidden_size, 32)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(32, 1)

    def forward(self, x):
        # Initialize hidden and cell states
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)

        out, _ = self.lstm(x, (h0, c0))   # out: (batch, seq_len, hidden_size)
        out = self.dropout(out[:, -1, :]) # take last time step
        out = self.relu(self.fc1(out))
        out = self.fc2(out)
        return out.squeeze(-1)

# Instantiate model
INPUT_SIZE  = len(feature_cols)  # 5 features
HIDDEN_SIZE = 64
NUM_LAYERS  = 2
DROPOUT     = 0.2

model = StockLSTM(INPUT_SIZE, HIDDEN_SIZE, NUM_LAYERS, DROPOUT).to(device)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"\nArchitecture rationale:")
print(f"  • LSTM({HIDDEN_SIZE}, layers={NUM_LAYERS})  : stacked LSTM captures multi-scale temporal patterns")
print(f"  • Dropout({DROPOUT})              : regularization — stock data easily overfits")
print(f"  • Dense({HIDDEN_SIZE}→32→1)           : regression head — outputs scaled next-day price")
print(f"  • batch_first=True            : input shape (batch, seq_len, features)")

## Step 5 — Train the Model

In [ ]:
LEARNING_RATE = 1e-3
EPOCHS        = 50
PATIENCE      = 7

criterion = nn.MSELoss()
optimizer = Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, min_lr=1e-6, verbose=True
)

# --- Training loop ---
def train_epoch(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        preds = model(x_batch)
        loss  = criterion(preds, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # gradient clipping
        optimizer.step()
        total_loss += loss.item() * len(x_batch)
    return total_loss / len(loader.dataset)

def eval_epoch(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            preds      = model(x_batch)
            total_loss += criterion(preds, y_batch).item() * len(x_batch)
    return total_loss / len(loader.dataset)

# --- Training ---
train_losses, val_losses = [], []
best_val_loss = float('inf')
patience_counter = 0

print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val Loss':>10}  {'LR':>10}")
print("-" * 45)

for epoch in range(1, EPOCHS + 1):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = eval_epoch(model, val_loader,   criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']
    if epoch % 5 == 0 or epoch == 1:
        print(f"{epoch:>6}  {train_loss:>12.6f}  {val_loss:>10.6f}  {current_lr:>10.2e}")

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        torch.save(model.state_dict(), 'best_stock_lstm.pth')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch} (best val loss: {best_val_loss:.6f})")
            break

print("\nTraining complete ✓")
print(f"Best model saved to best_stock_lstm.pth")

In [ ]:
# Plot training & validation loss
fig, ax = plt.subplots(figsize=(12, 5))
epochs_ran = range(1, len(train_losses) + 1)
ax.plot(epochs_ran, train_losses, label='Train Loss', color='#4e91d4', lw=2)
ax.plot(epochs_ran, val_losses,   label='Val Loss',   color='#e05c5c', lw=2, linestyle='--')
ax.set_title('Training & Validation Loss (MSE)', fontweight='bold')
ax.set_xlabel('Epoch'); ax.set_ylabel('MSE Loss (scaled)')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('stock_plot2_loss_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 2 saved ✓")

## Step 6 — Evaluate the Model

In [ ]:
# Load best model
model.load_state_dict(torch.load('best_stock_lstm.pth', map_location=device))
model.eval()
print("Best model loaded ✓")

# --- Collect predictions on test set ---
all_preds, all_true = [], []
with torch.no_grad():
    for x_batch, y_batch in test_loader:
        x_batch = x_batch.to(device)
        preds   = model(x_batch).cpu().numpy()
        all_preds.extend(preds)
        all_true.extend(y_batch.numpy())

all_preds = np.array(all_preds).reshape(-1, 1)
all_true  = np.array(all_true).reshape(-1, 1)

# Inverse-transform to real USD prices
preds_usd = scaler_y.inverse_transform(all_preds).flatten()
true_usd  = scaler_y.inverse_transform(all_true).flatten()

# --- Metrics ---
r2   = r2_score(true_usd, preds_usd)
rmse = np.sqrt(mean_squared_error(true_usd, preds_usd))
mae  = mean_absolute_error(true_usd, preds_usd)
mape = np.mean(np.abs((true_usd - preds_usd) / true_usd)) * 100

print(f"\n{'='*45}")
print(f"  TEST SET EVALUATION")
print(f"{'='*45}")
print(f"  R² Score  : {r2:.4f}")
print(f"  RMSE      : ${rmse:.2f}")
print(f"  MAE       : ${mae:.2f}")
print(f"  MAPE      : {mape:.2f}%")
print(f"{'='*45}")

# --- Plot: Predictions vs Actual ---
fig, axes = plt.subplots(2, 1, figsize=(14, 9))

# Full test set
axes[0].plot(true_usd,  label='Actual',    color='#4e91d4', lw=1.5)
axes[0].plot(preds_usd, label='Predicted', color='#e05c5c', lw=1.5, linestyle='--', alpha=0.85)
axes[0].set_title(f'{TICKER} — Full Test Set: Actual vs Predicted', fontweight='bold')
axes[0].set_xlabel('Trading Days'); axes[0].set_ylabel('Price (USD)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[0].text(0.02, 0.92, f'R² = {r2:.4f}  |  RMSE = ${rmse:.2f}  |  MAPE = {mape:.2f}%',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Zoom: last 60 trading days
axes[1].plot(true_usd[-60:],  label='Actual',    color='#4e91d4', lw=2)
axes[1].plot(preds_usd[-60:], label='Predicted', color='#e05c5c', lw=2, linestyle='--')
axes[1].set_title('Zoom: Last 60 Trading Days', fontweight='bold')
axes[1].set_xlabel('Trading Days'); axes[1].set_ylabel('Price (USD)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle(f'{TICKER} LSTM — Stock Price Prediction', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('stock_plot3_predictions.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 3 saved ✓")

# --- Scatter: Actual vs Predicted ---
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(true_usd, preds_usd, alpha=0.4, s=10, color='#4e91d4')
lim = [min(true_usd.min(), preds_usd.min()) * 0.98,
       max(true_usd.max(), preds_usd.max()) * 1.02]
axes[0].plot(lim, lim, 'r--', lw=1.5, label='Perfect prediction')
axes[0].set_title('Actual vs Predicted (scatter)', fontweight='bold')
axes[0].set_xlabel('Actual Price (USD)'); axes[0].set_ylabel('Predicted Price (USD)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

errors = true_usd - preds_usd
axes[1].hist(errors, bins=40, color='#5cb85c', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', lw=1.5)
axes[1].axvline(errors.mean(), color='orange', linestyle='--', lw=1.5, label=f'Mean error: ${errors.mean():.2f}')
axes[1].set_title('Prediction Error Distribution', fontweight='bold')
axes[1].set_xlabel('Error (USD)'); axes[1].set_ylabel('Count')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.suptitle('Model Error Analysis', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('stock_plot4_error_analysis.png', dpi=120, bbox_inches='tight')
plt.show()
print("Plot 4 saved ✓")

## Summary & Key Takeaways

**What I learned:**

1. **PyTorch Dataset & DataLoader pattern** — Subclassing `torch.utils.data.Dataset` with `__len__` and `__getitem__` lets us cleanly wrap sequence data. `DataLoader` handles batching, shuffling, and multi-process loading automatically.

2. **LSTM in PyTorch vs TensorFlow** — In PyTorch we manually initialize hidden states `(h0, c0)`, call `loss.backward()`, `optimizer.step()`, and toggle `model.train()` / `model.eval()`. More explicit but gives full control.

3. **Why gradient clipping?** — LSTM training can suffer from exploding gradients on financial time-series data. `nn.utils.clip_grad_norm_` caps the gradient norm at 1.0 to stabilize training.

4. **R² Score interpretation** — R² close to 1.0 means the model explains most variance in stock prices. On AAPL, LSTM achieves high R² because next-day price is strongly correlated with today's price (the model partially learns this lagged relationship).

5. **Scaler persistence** — Saving `scaler_X.pkl` and `scaler_y.pkl` alongside `best_stock_lstm.pth` ensures future inference uses the exact same normalization parameters as training.

6. **Train / Val / Test split for time series** — We split chronologically (not randomly) to avoid data leakage: the model never sees future data during training.